# 📈 Astoria Cleaning Services — Service Demand & Revenue Prediction (v2)
### Problem Statement: Predictive Revenue Modeling (Excluding Direct Cost Factors)

**Objective:** Build a regression model to predict `order_value_sgd` using high-level order characteristics.
- **Correction:** Per business requirements, `quantity` and `base_price_sgd` are excluded from the model to focus on identifying which services and locations naturally lead to higher revenue without relying on the raw invoice math.

**Business Impact Expected:**
- Identification of high-value zones and service categories.
- Better understanding of 'premium' demand drivers.

---

## 📋 Data Reference — Updated Columns 

| Column | Type | Purpose |
|--------|------|---------|
| `category` | Categorical | Broad item group |
| `service` | Categorical | Specific cleaning task |
| `zone` | Categorical | Customer location |
| `booking_day_of_week` | Categorical | Day the order was placed |
| `delicate_surcharge` | Numerical | Special handling costs |
| `express_multiplier` | Numerical | Speed premium |
| `order_value_sgd` | Numerical | **Target Variable** |


## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv('astoria_orders.csv')

# Fill missing values for multipliers
df['express_multiplier'] = df['express_multiplier'].fillna(1.0)

print(f'Dataset loaded: {df.shape[0]} rows.')

## 2. Feature Selection & Preprocessing
*Note: quantity and base_price_sgd have been removed to prevent data leakage and focus on contextual drivers.*

In [ ]:
features = ['category', 'service', 'delicate_surcharge', 'express_multiplier', 'zone', 'booking_day_of_week']
target = 'order_value_sgd'

X = df[features]
y = df[target]

cat_cols = ['category', 'service', 'zone', 'booking_day_of_week']
num_cols = ['delicate_surcharge', 'express_multiplier']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Preprocessing pipeline configured.')

## 3. Model Training & Evaluation

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Updated Model Performance:')
print(f'Mean Absolute Error: ${mae:.2f}')
print(f'R^2 Score: {r2:.4f}')

## 4. Visualizing Drivers (Excluding Price/Qty)

In [ ]:
# Feature Importance Plot
regressor = model.named_steps['regressor']
onehot_cols = model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_features = num_cols + list(onehot_cols)
importances = regressor.feature_importances_
indices = np.argsort(importances)[-15:] # Top 15 features

plt.figure(figsize=(10, 8))
plt.barh(range(len(indices)), importances[indices], color='teal', align='center')
plt.yticks(range(len(indices)), [all_features[i] for i in indices])
plt.xlabel('Importance')
plt.title('Top 15 Contextual Drivers of Order Value (Excl. Qty/Base Price)')
plt.tight_layout()
plt.savefig('updated_feature_importance.png')
plt.show()